In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [ ]:
import requests
from pathlib import Path

base_url = "https://oedi-data-lake.s3.amazonaws.com"

bronze_path = Path(
    "/lakehouse/default/Files/pvdaq/raw/telemetry/system_id=1199"
)

bronze_path.mkdir(parents=True, exist_ok=True)

for day in range(1, 8):

    file_name = f"system_1199__date_2010_10_{day:02d}.csv"

    file_url = (
        f"{base_url}/pvdaq/csv/pvdata/"
        f"system_id=1199/year=2010/month=10/"
        f"day={day}/{file_name}"
    )

    response = requests.get(file_url, timeout=60)
    response.raise_for_status()

    destination = bronze_path / file_name
    destination.write_bytes(response.content)

    print(f"Downloaded: {file_name}")

In [ ]:
# 1. Define the folder containing our seven CSV files
source_path = "/lakehouse/default/Files/pvdaq/raw/telemetry/system_id=1199/*.csv"

# 2. Read all CSV files into one PySpark DataFrame
df_telemetry = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(source_path)
)

# 3. Count the records
print("Total records:", df_telemetry.count())

# 4. Display the schema
df_telemetry.printSchema()

# 5. Display the first 10 records
display(df_telemetry.limit(10))

In [ ]:
from pathlib import Path

source_folder = Path(
    "/lakehouse/default/Files/pvdaq/raw/telemetry/system_id=1199"
)

print("Folder exists:", source_folder.exists())

print("CSV files:", len(list(source_folder.glob("*.csv"))))

In [ ]:
# 1. Define the local Lakehouse folder
source_path = "file:///lakehouse/default/Files/pvdaq/raw/telemetry/system_id=1199/*.csv"

# 2. Read the CSV files using PySpark
df_telemetry = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(source_path)
)

# 3. Verify the result
print("Total records:", df_telemetry.count())

df_telemetry.printSchema()

display(df_telemetry.limit(10))

In [ ]:
from pyspark.sql import functions as F

# 1. Calculate data quality metrics
df_quality = df_telemetry.agg(
    F.count("*").alias("total_records"),

    F.countDistinct("measured_on").alias("unique_timestamps"),

    F.sum(
        F.when(F.col("measured_on").isNull(), 1).otherwise(0)
    ).alias("missing_timestamps"),

    F.sum(
        F.when(F.col("ac_power__2714").isNull(), 1).otherwise(0)
    ).alias("missing_ac_power"),

    F.sum(
        F.when(F.col("dc_power__2715").isNull(), 1).otherwise(0)
    ).alias("missing_dc_power"),

    F.sum(
        F.when(F.col("ac_power__2714") < 0, 1).otherwise(0)
    ).alias("negative_ac_power"),

    F.min("measured_on").alias("first_measurement"),

    F.max("measured_on").alias("last_measurement")
)

# 2. Display quality results
display(df_quality)

In [ ]:
print("Spark version:", spark.version)

In [ ]:
# 1. Define the Bronze telemetry source
source_path = "file:///lakehouse/default/Files/pvdaq/raw/telemetry/system_id=1199/*.csv"

# 2. Load the seven CSV files
df_telemetry = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(source_path)
)

# 3. Verify the record count
print("Bronze telemetry records:", df_telemetry.count())

In [ ]:
# 1. Define the Silver Lakehouse destination
silver_path = "abfss://ac09d151-f58f-40f3-bf13-a61a3c24bcdb@onelake.dfs.fabric.microsoft.com/f63acb2c-3844-4da7-8866-f3b7c155a001/Tables"

# 2. Define the table name
table_name = "silver_pvdaq_telemetry"

# 3. Build the complete table path
table_path = f"{silver_path}/{table_name}"

# 4. Check whether the Delta table already exists
from delta.tables import DeltaTable

if not DeltaTable.isDeltaTable(spark, table_path):

    df_silver.write.format("delta").save(table_path)
    print("Silver Delta data saved successfully!")

else:
    print("Silver Delta data already exists. No overwrite performed.")

# 5. Verify the saved records
df_check = spark.read.format("delta").load(table_path)

print("Saved Silver records:", df_check.count())

In [ ]:
from pyspark.sql import functions as F

df_silver = (
    df_telemetry
    .select(
        F.col("system_id"),
        F.col("measured_on").alias("measurement_timestamp"),
        F.col("ac_power__2714").alias("ac_power_raw"),
        F.col("dc_power__2715").alias("dc_power_raw")
    )
    .withColumn(
        "measurement_date",
        F.to_date("measurement_timestamp")
    )
    .dropDuplicates(["system_id", "measurement_timestamp"])
)

print("Silver records:", df_silver.count())

display(df_silver.limit(10))

In [ ]:
# 1. Define the Silver Lakehouse destination
silver_path = "abfss://ac09d151-f58f-40f3-bf13-a61a3c24bcdb@onelake.dfs.fabric.microsoft.com/f63acb2c-3844-4da7-8866-f3b7c155a001/Tables"

# 2. Define the destination table name
table_name = "silver_pvdaq_telemetry"

# 3. Save the DataFrame in Delta format
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .save(f"{silver_path}/{table_name}")
)

print("Silver telemetry saved successfully!")

In [ ]:
df_check = spark.read.format("delta").load(
    f"{silver_path}/{table_name}"
)

print("Saved Silver records:", df_check.count())